# Eurostat Gap Discovery (Consumption and Prices)

Goal: automatically find Eurostat datasets related to consumption or prices that are not already covered by the current ENTSOG and FRED pulls.

What this notebook does:
1. Load existing FRED and ENTSOG coverage from `data/processed`.
2. Pull the full Eurostat table of contents via API (`eurostat` package, SDMX endpoint).
3. Filter to consumption/price themes.
4. Score likely overlap with already-fetched data.
5. Output prioritized candidate datasets to fetch next.

In [1]:
%pip install -q eurostat pandas numpy


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [6]:
from pathlib import Path
import re
import time
from difflib import SequenceMatcher

import numpy as np
import pandas as pd
import eurostat

ROOT = Path('/workspaces/high_frequency')
OUT = ROOT / 'data' / 'processed'

FRED_CATALOG = OUT / 'fred_europe_macro_energy_universe_catalog.csv'
ENTSOG_DAILY = OUT / 'entsog_daily_selected.csv'

EUROSTAT_CANDIDATES = OUT / 'eurostat_candidates_consumption_prices.csv'
EUROSTAT_NEW = OUT / 'eurostat_likely_new_consumption_prices.csv'
EUROSTAT_COVERED = OUT / 'eurostat_likely_covered_consumption_prices.csv'
TOC_CACHE = OUT / 'eurostat_toc_cache.csv'

TOC_FETCH_RETRIES = 4
TOC_FETCH_BACKOFF_SECONDS = 8

In [7]:
fred = pd.read_csv(FRED_CATALOG)
entsog = pd.read_csv(ENTSOG_DAILY)

print(f'FRED series: {len(fred):,}')
print(f'ENTSOG rows: {len(entsog):,}')
print('ENTSOG countries:', sorted(entsog['countryKey'].dropna().unique().tolist()))
print('ENTSOG indicators:', sorted(entsog['indicator'].dropna().unique().tolist()))

# Build a compact baseline text corpus for overlap checks
fred_text = (
    fred['id'].fillna('') + ' ' +
    fred['title'].fillna('') + ' ' +
    fred['sectors'].fillna('') + ' ' +
    fred['coverage_description'].fillna('')
).str.lower().tolist()

entsog_text = (
    entsog['indicator'].fillna('').astype(str) + ' ' +
    entsog['adjacentSystemsLabel'].fillna('').astype(str) + ' ' +
    entsog['operatorLabel'].fillna('').astype(str)
).str.lower().tolist()

baseline_text = fred_text + entsog_text
print(f'Baseline text entries: {len(baseline_text):,}')

FRED series: 515
ENTSOG rows: 48,048
ENTSOG countries: ['DE', 'ES', 'EU', 'FR', 'IT', 'NL', 'UK']
ENTSOG indicators: ['Allocation', 'Physical Flow']
Baseline text entries: 48,563


In [8]:
# Eurostat full table of contents (API through eurostat package)
# Under the hood: https://ec.europa.eu/eurostat/api/dissemination/sdmx/2.1/dataflow/all?format=JSON&compressed=true&lang=en

def fetch_toc_with_retries_and_cache(
    retries=TOC_FETCH_RETRIES,
    backoff=TOC_FETCH_BACKOFF_SECONDS,
    cache_path=TOC_CACHE,
    force_refresh=False,
    agency='EUROSTAT',
    lang='en',
):
    last_err = None

    # Fast path: reuse cache if available and refresh is not forced.
    if cache_path.exists() and not force_refresh:
        cached = pd.read_csv(cache_path)
        if len(cached) > 0:
            print(f'Loaded Eurostat TOC from cache: {cache_path} ({len(cached):,} rows)')
            return cached

    for attempt in range(1, retries + 1):
        try:
            print(f'Fetching Eurostat TOC (attempt {attempt}/{retries})...')
            toc_df = eurostat.get_toc_df(agency=agency, lang=lang)
            toc_df.columns = [c.strip().lower().replace(' ', '_') for c in toc_df.columns]

            # Persist cache for future runs.
            toc_df.to_csv(cache_path, index=False)
            print(f'Fetched and cached Eurostat TOC: {cache_path} ({len(toc_df):,} rows)')
            return toc_df
        except Exception as e:
            last_err = e
            if attempt < retries:
                wait_s = backoff * attempt
                print(f'Attempt {attempt} failed: {type(e).__name__}: {e}')
                print(f'Waiting {wait_s}s before retry...')
                time.sleep(wait_s)
            else:
                print(f'Final attempt failed: {type(e).__name__}: {e}')

    # Fallback to cache if network fetch failed.
    if cache_path.exists():
        cached = pd.read_csv(cache_path)
        print(f'Using stale TOC cache after fetch failure: {cache_path} ({len(cached):,} rows)')
        return cached

    raise RuntimeError(
        'Eurostat TOC fetch failed and no cache is available. '
        'Re-run later or reduce network load.'
    ) from last_err

# Set to True only when you explicitly want to refresh TOC from the API.
FORCE_TOC_REFRESH = False
toc = fetch_toc_with_retries_and_cache(force_refresh=FORCE_TOC_REFRESH)

# Expected columns include: title, code, type, data_start, data_end
display(toc.head(3))
print('Total Eurostat datasets in TOC:', len(toc))

Fetching Eurostat TOC (attempt 1/4)...
Attempt 1 failed: ReadTimeout: HTTPSConnectionPool(host='ec.europa.eu', port=443): Read timed out. (read timeout=120.0)
Waiting 8s before retry...
Fetching Eurostat TOC (attempt 2/4)...
Attempt 2 failed: ReadTimeout: HTTPSConnectionPool(host='ec.europa.eu', port=443): Read timed out. (read timeout=120.0)
Waiting 16s before retry...
Fetching Eurostat TOC (attempt 3/4)...
Fetched and cached Eurostat TOC: /workspaces/high_frequency/data/processed/eurostat_toc_cache.csv (8,209 rows)


,title,code,type,last_update_of_data,last_table_structure_change,data_start,data_end
0,Distribution of digital platform workers (at l...,LFST_DPW_10,dataset,2024-07-17T23:00:00+0200,2024-07-17T23:00:00+0200,2022,2022
1,Distribution of digital platform workers (at l...,LFST_DPW_11,dataset,2024-07-17T23:00:00+0200,2024-07-17T23:00:00+0200,2022,2022
2,Percentage of adults having a second job by nu...,LFST_HH2JCHI,dataset,2026-04-22T23:00:00+0200,2026-04-17T11:00:00+0200,2006,2025


Total Eurostat datasets in TOC: 8209


In [16]:
price_terms = [
    'price', 'prices', 'inflation', 'hicp', 'cpi', 'ppi',
    'producer price', 'consumer price', 'energy price',
    'electricity price', 'gas price', 'oil price', 'tariff'
]

consumption_terms = [
    'consumption', 'household consumption', 'final consumption',
    'expenditure', 'retail', 'demand', 'use of energy', 'energy use'
]

exclude_terms = [
    'metadata', 'classification', 'nomenclature', 'sdmx',
    'quality report', 'dictionary', 'documentation'
]

def compile_patterns(terms):
    # Word-boundary regex avoids false matches like 'enterprise' matching 'price'.
    return [re.compile(r'\b' + re.escape(t) + r'\b') for t in terms]

price_patterns = compile_patterns(price_terms)
consumption_patterns = compile_patterns(consumption_terms)
exclude_patterns = compile_patterns(exclude_terms)

def has_any(text, patterns):
    t = str(text).lower()
    return any(p.search(t) is not None for p in patterns)

def period_granularity(x):
    s = str(x).strip()
    if re.fullmatch(r'\d{4}-\d{2}', s):
        return 'month'
    if re.fullmatch(r'\d{4}', s):
        return 'year'
    if re.fullmatch(r'\d{4}-Q[1-4]', s):
        return 'quarter'
    if re.fullmatch(r'\d{4}-\d{2}-\d{2}', s):
        return 'day'
    return 'other'

def to_month_period_bounds(x):
    s = str(x).strip()
    if re.fullmatch(r'\d{4}-\d{2}', s):
        p = pd.Period(s, freq='M')
        return p, p
    if re.fullmatch(r'\d{4}', s):
        y = int(s)
        return pd.Period(f'{y}-01', freq='M'), pd.Period(f'{y}-12', freq='M')
    if re.fullmatch(r'\d{4}-Q[1-4]', s):
        y = int(s[:4])
        q = int(s[-1])
        q_start = 1 + (q - 1) * 3
        q_end = q_start + 2
        return pd.Period(f'{y}-{q_start:02d}', freq='M'), pd.Period(f'{y}-{q_end:02d}', freq='M')
    if re.fullmatch(r'\d{4}-\d{2}-\d{2}', s):
        p = pd.Period(s[:7], freq='M')
        return p, p
    return pd.NaT, pd.NaT

cand = toc.copy()
cand['title_l'] = cand['title'].fillna('').str.lower()
cand['is_price'] = cand['title_l'].apply(lambda x: has_any(x, price_patterns))
cand['is_consumption'] = cand['title_l'].apply(lambda x: has_any(x, consumption_patterns))
cand['is_excluded'] = cand['title_l'].apply(lambda x: has_any(x, exclude_patterns))

cand = cand[(cand['is_price'] | cand['is_consumption']) & (~cand['is_excluded'])].copy()
cand['theme'] = np.select(
    [cand['is_price'] & cand['is_consumption'], cand['is_price'], cand['is_consumption']],
    ['price+consumption', 'price', 'consumption'],
    default='other'
)

# FRED-style lock criteria for Eurostat candidate datasets
cand['start_granularity'] = cand['data_start'].apply(period_granularity)
cand['end_granularity'] = cand['data_end'].apply(period_granularity)
cand[['start_period_min', 'start_period_max']] = cand['data_start'].apply(
    lambda x: pd.Series(to_month_period_bounds(x))
)
cand[['end_period_min', 'end_period_max']] = cand['data_end'].apply(
    lambda x: pd.Series(to_month_period_bounds(x))
)

# Monthly criterion: explicit monthly metadata in period fields or title.
cand['is_monthly'] = (
    (cand['start_granularity'] == 'month')
    | (cand['end_granularity'] == 'month')
    | cand['title_l'].str.contains(r'\bmonthly\b', regex=True)
    | cand['title_l'].str.contains(r'\bmonth\b', regex=True)
)

target_start = pd.Period('2018-01', freq='M')
target_end = pd.Period('2026-03', freq='M')

cand['starts_pre_2018'] = cand['start_period_min'].notna() & (cand['start_period_min'] <= target_start)
cand['ends_at_least_2026_03'] = cand['end_period_max'].notna() & (cand['end_period_max'] >= target_end)

cand = cand[
    cand['is_monthly'] & cand['starts_pre_2018'] & cand['ends_at_least_2026_03']
] .copy()

print('Candidate Eurostat datasets after lock criteria (monthly, start<=2018-01, end>=2026-03):', len(cand))
cand[['code', 'title', 'theme', 'data_start', 'data_end', 'is_monthly']].head(10)

Candidate Eurostat datasets after lock criteria (monthly, start<=2018-01, end>=2026-03): 18


,code,title,theme,data_start,data_end,is_monthly
1511,NRG_CB_GASM,"Supply, transformation and consumption of gas ...",consumption,2008-01,2026-03,True
1812,PRC_FSC_IDX,Food price monitoring tool,price,2005-01,2026-03,True
1846,PRC_HICP_CT,Harmonised index of consumer prices (HICP) - E...,price,2002-12,2026-03,True
1848,PRC_HICP_CTR,Harmonised index of consumer prices (HICP) - E...,price,2002-01,2026-03,True
1858,PRC_HICP_FPD,Harmonised index of consumer prices (HICP) - E...,price,1996-01,2026-03,True
1871,PRC_HICP_MINR,Harmonised index of consumer prices (HICP) - E...,price,1996-01,2026-03,True
1911,PRC_IPC_G20,G20 CPI all-items - Group of Twenty - Consumer...,price,1996-01,2026-03,True
5154,EI_BSRT_M_R2,Retail trade confidence indicator and survey r...,consumption,1984-02,2026-04,True
5163,EI_CPHI_M,Harmonised index of consumer prices - monthly ...,price,1996-01,2026-03,True
5189,EI_ISRR_M,Retail trade growth rates by NACE Rev. 2 activ...,consumption,1991-02,2026-03,True


In [17]:
# Overlap scoring against what is already covered by FRED/ENTSOG
overlap_hard_terms = [
    'harmonized index of consumer prices', 'hicp', 'brent',
    'physical flow', 'allocation', 'gas flow', 'nasdaq',
    'marginal lending facility'
]
overlap_hard_patterns = compile_patterns(overlap_hard_terms)

def best_similarity(title, corpus, sample_every=120):
    # Speed-up: sample corpus for broad screening, then refine if needed
    sampled = corpus[::sample_every] if len(corpus) > sample_every else corpus
    t = str(title).lower()
    if not sampled:
        return 0.0
    return max(SequenceMatcher(None, t, c).ratio() for c in sampled)

cand['hard_overlap'] = cand['title_l'].apply(lambda t: has_any(t, overlap_hard_patterns))
cand['sim_to_existing'] = cand['title'].fillna('').apply(lambda x: best_similarity(x, baseline_text))

# Heuristic classification
cand['likely_covered'] = cand['hard_overlap'] | (cand['sim_to_existing'] >= 0.72)
cand['priority_score'] = (
    (1 - cand['sim_to_existing'])
    + cand['is_consumption'].astype(int) * 0.15
    + cand['is_price'].astype(int) * 0.10
)

cand = cand.sort_values(['likely_covered', 'priority_score'], ascending=[True, False]).reset_index(drop=True)

new_df = cand[~cand['likely_covered']].copy()
covered_df = cand[cand['likely_covered']].copy()

print('Likely NEW datasets:', len(new_df))
print('Likely already covered datasets:', len(covered_df))
print()
print('Top likely NEW candidates:')
preview_cols = ['code', 'title', 'theme', 'data_start', 'data_end', 'sim_to_existing']
display(new_df.reindex(columns=preview_cols).head(30))

Likely NEW datasets: 14
Likely already covered datasets: 4

Top likely NEW candidates:


,code,title,theme,data_start,data_end,sim_to_existing
0,STS_TRTU_M,Turnover and volume of sales in wholesale and ...,consumption,1991-01,2026-03,0.319328
1,EI_ISRR_M,Retail trade growth rates by NACE Rev. 2 activ...,consumption,1991-02,2026-03,0.330097
2,EI_ISRT_M,Retail trade index by NACE Rev. 2 activity - m...,consumption,1991-01,2026-03,0.372549
3,EI_BSRT_M_R2,Retail trade confidence indicator and survey r...,consumption,1984-02,2026-04,0.373832
4,STS_TRLB_M,Labour input in wholesale and retail trade - m...,consumption,1994-01,2026-03,0.400000
5,STS_INPPD_M,"Producer prices in industry, domestic market -...",price,1975-01,2026-03,0.378947
6,STS_INPPND_M,"Producer prices in industry, non domestic mark...",price,1962-01,2026-03,0.380952
7,STS_INPI_M,Import prices in industry - monthly data,price,1962-01,2026-03,0.383562
8,STS_INPP_M,"Producer prices in industry, total - monthly data",price,1976-01,2026-03,0.387097
9,PRC_FSC_IDX,Food price monitoring tool,price,2005-01,2026-03,0.416667


In [18]:
# Persist outputs for downstream automation
cols = [
    'code', 'title', 'type', 'last_update_of_data', 'last_table_structure_change',
    'data_start', 'data_end', 'theme', 'sim_to_existing', 'hard_overlap',
    'likely_covered', 'priority_score'
]

cand[cols].to_csv(EUROSTAT_CANDIDATES, index=False)
new_df[cols].to_csv(EUROSTAT_NEW, index=False)
covered_df[cols].to_csv(EUROSTAT_COVERED, index=False)

print('Saved:')
print('-', EUROSTAT_CANDIDATES)
print('-', EUROSTAT_NEW)
print('-', EUROSTAT_COVERED)

Saved:
- /workspaces/high_frequency/data/processed/eurostat_candidates_consumption_prices.csv
- /workspaces/high_frequency/data/processed/eurostat_likely_new_consumption_prices.csv
- /workspaces/high_frequency/data/processed/eurostat_likely_covered_consumption_prices.csv


In [ ]:
# Optional: quick API sample pull on top likely-new datasets
RUN_SAMPLE_PULL = False
TOP_N = 10

if RUN_SAMPLE_PULL:
    sample_codes = new_df['code'].dropna().head(TOP_N).tolist()
    sample_results = []
    for code in sample_codes:
        try:
            tmp = eurostat.get_data_df(
                code,
                flags=False,
                filter_pars={'startPeriod': 2018}
            )
            n_rows = 0 if tmp is None else len(tmp)
            sample_results.append({'code': code, 'status': 'ok', 'rows': n_rows})
        except Exception as e:
            sample_results.append({'code': code, 'status': 'error', 'rows': 0, 'error': str(e)[:180]})

    sample_df = pd.DataFrame(sample_results)
    display(sample_df)
else:
    print('Sample pull disabled. Set RUN_SAMPLE_PULL=True to test live pulls for top candidates.')

## Notes
- This notebook performs table-level discovery, not full-series extraction yet.
- Next step after validation: build a fetch notebook that loops through `eurostat_likely_new_consumption_prices.csv` and applies dataset-specific filters (geo, unit, frequency).
- Overlap detection is heuristic and intentionally conservative.